## 1. Validar classificação regex

Primeiro teste crítico.

In [0]:
from pyspark.sql.functions import lit

from src.ml.features.tema import (
    classificar_tema
)

spark.range(1).select(

    classificar_tema(

        lit(
            "Institui programa nacional de vacinação infantil."
        )

    )["tema"].alias("tema"),

    classificar_tema(

        lit(
            "Institui programa nacional de vacinação infantil."
        )

    )["score_max"].alias("score_max"),

    classificar_tema(

        lit(
            "Institui programa nacional de vacinação infantil."
        )

    )["score_second"].alias("score_second"),

    classificar_tema(

        lit(
            "Institui programa nacional de vacinação infantil."
        )

    )["score_margem"].alias("margem"),

    classificar_tema(

        lit(
            "Institui programa nacional de vacinação infantil."
        )

    )["confianca"].alias("confianca")

).show(
    truncate=False
)

## 2. Gerar base treino nova

In [0]:
from src.ml.training.base_generator import (
    gerar_base_treino
)

from src.ml.config.training_configs import (
    CONFIG_TEMA
)

df_base = gerar_base_treino(
    CONFIG_TEMA
)

print(
    f"Total registros: {df_base.count()}"
)

df_base.groupBy(
    "tema_ementa"
).count().orderBy(
    "count",
    ascending=False
).show(
    truncate=False
)

In [0]:
from pyspark.sql.functions import col

df_base.select(
    "ementa",
    "tema_ementa"
).show(
    50,
    truncate=False
)

## 3. Treinar modelo

In [0]:
from src.ml.training.trainer import (
    executar_treino
)

resultado = executar_treino(

    df=df_base,

    target_col="tema_ementa"
)


## 4. Ver métricas

In [0]:
import pandas as pd

pd.DataFrame(
    resultado["report"]
).T


## 5. Ver matriz de confusão

In [0]:
resultado[
    "confusion_matrix"
]

### Salvar erros do modelo

In [0]:
erros = pd.DataFrame({

    "texto": resultado["X_test"],

    "real": resultado["y_test"],

    "predito": resultado["predictions"]

})

erros = erros[
    erros["real"] != erros["predito"]
]

In [0]:
erros.head(50)

In [0]:
import pandas as pd

erros = pd.DataFrame({

    "texto": resultado["X_test"],

    "real": resultado["y_test"],

    "predito": resultado["predictions"]

})

erros = erros[
    erros["real"] != erros["predito"]
]

erros.head(100)

In [0]:
from pyspark.sql.functions import col, rand

df_gold_review = (
    df_base
    .select(
        "ementa",
        "tema_ementa"
    )
    .withColumnRenamed(
        "tema_ementa",
        "tema_regex"
    )
    .orderBy(rand(seed=42))
    .limit(300)
)

df_gold_review.show(
    20,
    truncate=False
)

In [0]:
output_path = "/dbfs/FileStore/gold_review_temas.csv"

(
    df_gold_review
    .toPandas()
    .assign(
        tema_humano="",
        observacao=""
    )
    .to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig"
    )
)

print(output_path)

## 6. Registrar modelo


### Temas

In [0]:
from src.ml.orchestration.training_runner import (
    executar_pipeline_treino
)

from src.ml.config.training_configs import (
    CONFIG_TEMA
)

executar_pipeline_treino(
    CONFIG_TEMA
)

#### 1. Validar se o modelo carregou

In [0]:
from src.ml.inference.udf_loader import (
    criar_udf_classificacao
)

tema_ml = criar_udf_classificacao(
    model_name="api_dados_abertos.ml.tema_classificador",
    fallback="Tema Não Explícito"
)

#### 2. Testar inferência **ML**

In [0]:
import mlflow.sklearn

model_uri = "models:/api_dados_abertos.ml.tema_classificador@champion"

model = mlflow.sklearn.load_model(
    model_uri
)

model.predict([
    "Institui programa nacional de vacinação infantil."
])

In [0]:
from pyspark.sql.functions import lit

spark.range(1).select(
    tema_ml(
        lit("Institui programa nacional de vacinação infantil.")
    ).alias("tema_ml")
).show(
    truncate=False
)

### Natureza Jurídica

In [0]:
from src.ml.orchestration.training_runner import (
    executar_pipeline_treino
)

from src.ml.config.training_configs import (
    CONFIG_NATUREZA
)

executar_pipeline_treino(
    CONFIG_NATUREZA
)

#### 1. Validar se o modelo carregou

In [0]:
natureza_ml = criar_udf_classificacao(
    model_name="api_dados_abertos.ml.natureza_classificador",
    model_alias="champion",
    fallback="Outros Tipos"
)

#### 2. Testar inferência ML

In [0]:
spark.range(1).select(
    natureza_ml(
        lit("Altera a Lei nº 8.080, para dispor sobre vacinação infantil.")
    ).alias("natureza_ml")
).show(
    truncate=False
)

## 7. Testar transformação gold sem gravar

In [0]:
from src.config.project_config import (
    STORAGE_CONFIG
)

from src.utils.storage.delta_io import (
    read_table
)

from src.gold.transforms.proposicoes import (
    transform_proposicoes as transform_gold
)

df_silver = read_table(
    spark,
    STORAGE_CONFIG,
    "silver",
    "proposicoes"
)

df_gold_test = transform_gold(
    df_silver
)

df_gold_test.printSchema()

## 8. Validar distribuição das classificações

In [0]:
df_gold_test.groupBy(
    "tema_ementa"
).count().orderBy(
    "count",
    ascending=False
).show(
    truncate=False
)

In [0]:
df_gold_test.groupBy(
    "natureza_juridica"
).count().orderBy(
    "count",
    ascending=False
).show(
    truncate=False
)

In [0]:
df_gold_test.groupBy(
    "tipo_documental"
).count().orderBy(
    "count",
    ascending=False
).show(
    truncate=False
)

## 9. Ver amostra qualitativa

In [0]:
df_gold_test.select(
    "ementa",
    "tema_ementa",
    "natureza_juridica",
    "tipo_documental",
    "categoria_regimental",
    "peso_regimental",
    "flag_normativa",
    "flag_fiscalizacao",
    "flag_baixo_impacto"
).show(
    50,
    truncate=False
)

In [0]:
from pyspark.sql.functions import col
df_gold_test.filter(
    col("origem_tema") == "ML"
).select(
    "ementa",
    "tema_ementa"
).show(
    50,
    truncate=False
)